# NIFTY-50 Investment Intelligence — Full Training on Kaggle GPU

Runs the complete offline pipeline on the **real** competition datasets and produces every submission artifact (model comparison, forecasts, recommendations, risk tables, EDA charts, PDF report), plus GPU-accelerated LSTM/Transformer training.

**Setup (one-time, in the right sidebar):**
1. *Input → Add Input* → search and attach **`rohanrao/nifty50-stock-market-data`** (primary provided dataset)
2. Optionally also attach **`stoicstatic/india-stock-data-nse-1990-2020`** (the second provided dataset — extends history back to 1990 for overlapping symbols; auto-skipped if absent)
3. *Input → Add Input → Upload* → upload `investment-intelligence-kaggle.zip` as a private dataset (name it e.g. `nifty-intel-code`) and attach it
4. *Settings → Accelerator* → **GPU T4 x2** (or P100)
5. Run all cells. Artifacts land in `/kaggle/working/output/` — downloadable from the Output tab.

In [ ]:
# 1) Locate inputs and unpack the project code.
# Kaggle AUTO-EXTRACTS uploaded zips, so the code dataset may contain
# either the zip itself or the already-extracted folders — handle both.
import glob, os, shutil, zipfile, pathlib

print('attached inputs:', os.listdir('/kaggle/input'))

PROJECT = pathlib.Path('/kaggle/working/investment-intelligence')

if not (PROJECT / 'ml').exists():
    zips = [z for z in glob.glob('/kaggle/input/*/**/*.zip', recursive=True)
            if 'investment-intelligence' in os.path.basename(z).lower()]
    if zips:
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall('/kaggle/working')
    else:
        markers = glob.glob('/kaggle/input/*/**/ml/utils.py', recursive=True)
        assert markers, ('Project code not found — attach the dataset you created by '
                         'uploading investment-intelligence-kaggle.zip')
        src = pathlib.Path(markers[0]).parent.parent
        shutil.copytree(src, PROJECT, dirs_exist_ok=True)
assert (PROJECT / 'ml').exists(), 'project layout not recognized'

# Find the primary NIFTY-50 dataset wherever Kaggle mounted it: the folder
# holding RELIANCE.csv, preferring the one that also has stock_metadata.csv
# (distinguishes it from the 1990-2020 additional dataset).
hits = glob.glob('/kaggle/input/**/RELIANCE.csv', recursive=True)
assert hits, ('NIFTY dataset not found under /kaggle/input — attach '
              'rohanrao/nifty50-stock-market-data via Input → Add Input')

def _score(path):
    d = os.path.dirname(path)
    return (('nifty50' in path.lower()) * 2
            + os.path.exists(os.path.join(d, 'stock_metadata.csv')))

primary_dir = os.path.dirname(max(hits, key=_score))
print('primary dataset dir:', primary_dir)

nifty_csvs = [c for c in glob.glob(f'{primary_dir}/*.csv')
              if os.path.basename(c) != 'nifty50_all.csv']
raw = PROJECT / 'data' / 'raw'
raw.mkdir(parents=True, exist_ok=True)
for c in nifty_csvs:
    shutil.copy2(c, raw / os.path.basename(c))
print(f'project at {PROJECT}, {len(list(raw.glob("*.csv")))} CSVs staged')

In [ ]:
# 1b) OPTIONAL — extend history with the second provided dataset
# (stoicstatic/india-stock-data-nse-1990-2020). If attached, NIFTY symbols
# that overlap get their pre-2000 history prepended (~1 extra decade of
# training data). Safely skipped when the dataset isn't attached.
import pandas as pd

extra_csvs = [c for c in glob.glob('/kaggle/input/*/**/*.csv', recursive=True)
              if 'india-stock-data' in c.lower() or 'nse-1990' in c.lower()]
if not extra_csvs:
    print('additional dataset not attached — skipping (platform works fine without it)')
else:
    by_symbol = {}
    for c in extra_csvs:
        key = os.path.splitext(os.path.basename(c))[0].upper()
        key = key.replace('_DATA', '').replace('-EQ', '').replace('__EQ__', '')
        by_symbol.setdefault(key, c)
    extended = 0
    for primary_path in sorted(raw.glob('*.csv')):
        sym = primary_path.stem.upper()
        src = by_symbol.get(sym)
        if sym == 'STOCK_METADATA' or not src:
            continue
        try:
            extra = pd.read_csv(src)
            extra.columns = [str(col).strip().title() for col in extra.columns]
            if not {'Date', 'Open', 'High', 'Low', 'Close', 'Volume'}.issubset(extra.columns):
                continue
            primary = pd.read_csv(primary_path)
            extra['Date'] = pd.to_datetime(extra['Date'], errors='coerce')
            primary['Date'] = pd.to_datetime(primary['Date'], errors='coerce')
            older = extra[extra['Date'] < primary['Date'].min()].dropna(subset=['Date'])
            if older.empty:
                continue
            older = older.assign(Symbol=sym, Series='EQ')
            merged = pd.concat([older, primary], ignore_index=True, sort=False)
            merged = merged.sort_values('Date').drop_duplicates('Date', keep='last')
            merged.to_csv(primary_path, index=False)
            extended += 1
        except Exception as exc:
            print(f'{sym}: skipped ({exc})')
    print(f'extended {extended} symbols with pre-2000 history')

In [ ]:
# 2) Dependencies (Kaggle images ship most of these; this fills the gaps)
%pip install -q fastapi uvicorn lightgbm shap lime pyarrow
import tensorflow as tf
print('TF', tf.__version__, '— GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 3) Environment: real data only, no synthetic fallback
import os, sys
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
os.environ['ALLOW_SYNTHETIC_DATA'] = 'false'
os.environ['DATA_DIR'] = str(PROJECT / 'data')
os.environ['ARTIFACTS_DIR'] = str(PROJECT / 'ml' / 'artifacts')

In [ ]:
# 4) Ingestion: validate -> clean -> parquet
from backend.data_loader import run_pipeline
panel = run_pipeline()
panel.groupby('Symbol').size().describe()

In [ ]:
# 5) Full offline pipeline: EDA + model comparison + forecasts +
#    recommendations + risk tables + PDF report (classical models, ~CPU)
!python scripts/train_all.py 2>&1 | tail -25

In [ ]:
# 6) GPU showcase: walk-forward LSTM & Transformer vs best classical model
import pandas as pd
from ml.training.trainer import walk_forward_evaluate

DEEP_SYMBOLS = ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY']
rows = []
for sym in DEEP_SYMBOLS:
    df = panel[panel['Symbol'] == sym].sort_values('Date')
    for model in ('lstm', 'transformer', 'lightgbm'):
        try:
            res = walk_forward_evaluate(df, model, horizon=20, n_splits=3)
            rows.append({'symbol': sym, 'model': model, **res.metrics})
        except Exception as exc:
            print(f'{sym}/{model} failed: {exc}')
deep_cmp = pd.DataFrame(rows).sort_values(['symbol', 'rmse'])
deep_cmp.to_csv('ml/artifacts/deep_model_comparison.csv', index=False)
deep_cmp

In [ ]:
# 7) Package every artifact for download
import shutil, pathlib
out = pathlib.Path('/kaggle/working/output')
out.mkdir(exist_ok=True)
shutil.copytree(PROJECT / 'ml' / 'artifacts', out / 'artifacts', dirs_exist_ok=True)
shutil.copytree(PROJECT / 'reports' / 'generated', out / 'reports', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/nifty_intel_artifacts', 'zip', out)
print('Done — download nifty_intel_artifacts.zip from the Output tab')
for p in sorted(out.rglob('*')):
    if p.is_file():
        print(f'{p.relative_to(out)}  ({p.stat().st_size//1024} KB)')